## Pré-processamento

Etapas de limpeza e preparação dos dados para modelagem.
Cada decisão foi fundamentada na EDA anterior.

#### 2.1 Remoção de var14
Coluna removida por contaminação sistêmica via ffill (10.8% dos dados).

#### 2.2 Remoção de linhas anômalas
Removidas linhas onde TARGET < 5, var11 = 0 ou var12 = 0.
Esses valores indicam parada de processo ou falha de sensor —
não representam operação real e contaminariam o modelo.

#### 2.3 Criação do alvo com antecedência de 1 hora
df['y'] = df['TARGET'].shift(-60)
Cada linha passa a ter como alvo o KPI que ocorrerá 60 minutos depois.
Isso implementa a previsão com 1h de antecedência exigida no desafio.

#### 2.4 Separação de features e alvo
X = sensores de processo | y = TARGET futura (t+60min)

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

dados_novo = pd.read_csv('dados_planta_limpos.csv')

print("Shape carregado:", dados_novo.shape)
print("Colunas:", dados_novo.columns.tolist())

Shape carregado: (132478, 15)
Colunas: ['TS', 'var1', 'var2', 'var3', 'var4', 'var5', 'var6', 'TARGET', 'var8', 'var9', 'var10', 'var11', 'var12', 'var13', 'var15']


In [3]:
# 2.2 Remover linhas anômalas
antes = len(dados_novo)
dados_novo = dados_novo[dados_novo['TARGET'] >= 5]
dados_novo = dados_novo[dados_novo['var11'] != 0]
dados_novo = dados_novo[dados_novo['var12'] != 0]
depois = len(dados_novo)
print(f"Linhas removidas: {antes - depois}")
print(f"Linhas restantes: {depois}")

# 2.3 Criar o alvo — o coração do projeto
dados_novo['y'] = dados_novo['TARGET'].shift(-60)
dados_novo = dados_novo.dropna(subset=['y'])
print(f"Shape após shift: {dados_novo.shape}")

# 2.4 Separar X e y
features = [col for col in dados_novo.columns 
            if col not in ['TARGET', 'y', 'TS']]
X = dados_novo[features]
y = dados_novo['y']

print(f"\nFeatures ({len(features)}): {features}")
print(f"Shape X: {X.shape}")
print(f"Shape y: {y.shape}")

Linhas removidas: 1255
Linhas restantes: 131223
Shape após shift: (131163, 16)

Features (13): ['var1', 'var2', 'var3', 'var4', 'var5', 'var6', 'var8', 'var9', 'var10', 'var11', 'var12', 'var13', 'var15']
Shape X: (131163, 13)
Shape y: (131163,)


In [5]:
print("Colunas antes de salvar:", dados_novo.columns.tolist())
print("Shape antes de salvar:", dados_novo.shape)

dados_novo.to_csv('dados_planta_modelagem.csv', index=False)
print("Salvo com sucesso.")

Colunas antes de salvar: ['TS', 'var1', 'var2', 'var3', 'var4', 'var5', 'var6', 'TARGET', 'var8', 'var9', 'var10', 'var11', 'var12', 'var13', 'var15', 'y']
Shape antes de salvar: (131163, 16)
Salvo com sucesso.


## Pré-processamento — 2.4 Resultado final

- Linhas removidas (zeros e paradas): 1.255
- Linhas restantes para treino: 131.163
- Features de entrada (X): 13 variáveis de processo
- Alvo (y): TARGET deslocada 60 minutos à frente
- Shape X: (131.163, 13) | Shape y: (131.163,)

Os shapes de X e y são iguais — dados prontos para modelagem.